# Slater Determinats (SD), Clebsch-Gordan (CG) coefficients and Configuration State Functions (CSFs)


## Table of Content: <a name="TOC"></a>

1. [Generating determinants](#1)

   1.1. [Main function](#1.1)
   
   1.2. [Helper functions](#1.2)
   
2. [Clebsch-Gordan Coefficients: Theory and Role in Spin-Adapted CI](#2)

   2.1. [Physical meaning of Clebsch–Gordan coefficients](#2.1)
   
   2.2. [Why CG coefficients appear in electronic structure](#2.2)
   
   2.3. [Basis transformation](#2.3)
   
   2.4. [Selection rules (why many coefficients are zero)](#2.4)
   
   2.5. [Racah formula (explicit expression)](#2.5)
   
   2.6. [Half-integer spins and implementation detail](#2.6)
   
   2.7. [Explicit spin-½ example (two electrons)](#2.7)
   
   2.8. [Relation to CSFs and the T matrix](#2.8)   

   2.9. [Numerical considerations](#2.9)
   
   2.10. [Main function](#2.10)
   
   2.11. [Helper functions](#2.11)
   
3. [Constructing configuration state functions (CSFs)](#3)

   3.1. [Main function](#3.1)
   
   3.2. [Helper functions](#3.2)
   

## A. Learning objectives

- To gain practical skills in generating SDs 
- To observe how sign of SD depends on the ordering of the corresponding spin-orbitals
- To identify the importance and physical meaning of Clebsch-Gordan coefficients
- To be able to compute numerical values of Clebsch-Gordan coefficients for given spin-coupling situation
- To be able to construct and print CSFs for given S and Ms constraints


## B. Use cases

- Manually construct a Slater Determinant basis
- Constructing configuration spin functions
- Spin-adaptation of Slater determinants
- Compute Clebsch-Gordan coefficients

## C. Functions

- `libra_py`
  - `citools`
    - `clebsch_gordan`
      - [`clebsch_gordan`](#clebsch_gordan-1)
      - [`recursive_couple_spins_int`](#recursive_couple_spins_int-1)
    - `csf`
      - [`build_unpaired_spinlist`](#build_unpaired_spinlist-1)
      - [`generate_CSFs_grouped`](#generate_CSFs_grouped-1)
      - [`group_key_from_det`](#group_key_from_det-1)
      - [`print_csfs`](#print_csfs-1)
    - `slatdet`
      - [`canonical_sort_key`](#canonical_sort_key-1)
      - [`generate_determinants_with_parity`](#generate_determinants_with_parity-1)
      - [`permutation_parity`](#permutation_parity-1)      

In [1]:
import libra_py.citools.clebsch_gordan as cg
import libra_py.citools.slatdet as sd
import libra_py.citools.csf as csf

## 1. Generating determinants
[Back to TOC](#TOC)
<a name="1"></a>

A more detailed discussion of the conventions and the corresponding math is 
given in the [next tutorial](../2_slatdet_and_interfaces/tutorial.ipynb)

### 1.1. Main function
[Back to TOC](#TOC)
<a name="1.1"></a>

<a name="generate_determinants_with_parity-1"></a>

In [2]:
help(sd.generate_determinants_with_parity)

Help on function generate_determinants_with_parity in module libra_py.citools.slatdet:

generate_determinants_with_parity(active_orbitals: 'List[int]', N: 'int', allow_double_occupancy: 'bool' = True) -> 'Generator[Tuple[Tuple[int, ...], int], None, None]'
    Generate all possible Slater determinants (configurations) from a given set of 
    active orbitals, including their permutation parities.
    
    Each spatial orbital in `active_orbitals` contributes two spin-orbitals: 
    α (represented by +i) and β (represented by -i). The function yields all 
    unique combinations of `N` spin-orbitals consistent with the Pauli principle 
    and, optionally, with or without double occupancy of the same spatial orbital.
    
    The permutation parity is computed relative to the canonically sorted order 
    of spin-orbitals, allowing subsequent antisymmetrization when constructing 
    configuration state functions (CSFs).
    
    Parameters
    ----------
    active_orbitals : list of i

In [3]:
active_orbitals = [1, 2]
N = 2

determinants = list(sd.generate_determinants_with_parity(active_orbitals, N))
print(f"Active orbitals: {active_orbitals}")
print(f"Number of electrons: {N}")
print("Generated Slater determinants and their parity:")
for det in determinants:
    print(det)

Active orbitals: [1, 2]
Number of electrons: 2
Generated Slater determinants and their parity:
((1, -1), 1)
((1, 2), 1)
((1, -2), 1)
((-1, 2), 1)
((-1, -2), 1)
((2, -2), 1)


In [4]:
determinants = list(sd.generate_determinants_with_parity(active_orbitals, N, allow_double_occupancy=False))
print(f"Active orbitals: {active_orbitals}")
print(f"Number of electrons: {N}")
print("Generated Slater determinants and their parity:")
for det in determinants:
    print(det)

Active orbitals: [1, 2]
Number of electrons: 2
Generated Slater determinants and their parity:
((1, 2), 1)
((1, -2), 1)
((-1, 2), 1)
((-1, -2), 1)


### 1.2. Helper functions
[Back to TOC](#TOC)
<a name="1.2"></a>

The above function that generates determinants utilizes the following helper functions:
<a name="permutation_parity-1"></a>

In [5]:
help(sd.permutation_parity)

Help on function permutation_parity in module libra_py.citools.slatdet:

permutation_parity(original: 'Sequence[Any]', sorted_target: 'Sequence[Any]') -> 'int'
    Compute the permutation parity (+1 or -1) that maps one sequence to another.
    
    Given two sequences containing the same unique elements, this function determines 
    whether the permutation required to reorder `original` into `sorted_target` is 
    even or odd. It does so by constructing the index mapping between the two sequences 
    and counting the number of inversions.
    
    Parameters
    ----------
    original : sequence of Any
        The starting ordering of elements. Typically represents a specific configuration
        of spin-orbitals or basis functions.
    sorted_target : sequence of Any
        The target ordering containing the same elements as `original`, but arranged
        in the desired canonical order (e.g., from `canonical_sort_key`).
    
    Returns
    -------
    int
        The parity 

In [6]:
sd.permutation_parity((2, 1), (1, 2))

-1

In [7]:
sd.permutation_parity((3, 1, 2), (1, 2, 3))

1

In [8]:
sd.permutation_parity((1, 2, 3), (1, 2, 3))

1

<a name="canonical_sort_key-1"></a>

In [9]:
help(sd.canonical_sort_key)

Help on function canonical_sort_key in module libra_py.citools.slatdet:

canonical_sort_key(x: 'int') -> 'tuple[int, int]'
    Canonical sorting key for spin-orbitals.
    
    This function defines a consistent ordering of spin-orbitals used to 
    canonicalize Slater determinants. It ensures that spin-orbitals are 
    first ordered by the absolute value of their orbital index (spatial orbital),
    and then by spin: α-spin (positive index) precedes β-spin (negative index).
    
    Parameters
    ----------
    x : int
        Spin-orbital label, where positive values (e.g., +1, +2, +3) represent 
        α-spin orbitals and negative values (e.g., -1, -2, -3) represent β-spin orbitals.
    
    Returns
    -------
    tuple of (int, int)
        Sorting key `(abs(x), spin_order)` where:
        
        - `abs(x)` ensures grouping by spatial orbital index.  
        - `spin_order` is 0 for α-spin (x > 0) and 1 for β-spin (x < 0), 
          so that α-spin comes before β-spin in can

In [10]:
sorted([+2, -1, +1, -2], key=sd.canonical_sort_key)

[1, -1, 2, -2]

In [11]:
sd.canonical_sort_key(-3)

(3, 1)

## 2. Clebsch-Gordan Coefficients: Theory and Role in Spin-Adapted CI
[Back to TOC](#TOC)
<a name="2"></a>

Here, we briefly explains the **theoretical background and practical meaning** of the function

`clebsch_gordan(j1, j2, J, m1, m2, M)`

which computes Clebsch–Gordan (CG) coefficients using the **Racah formula**. These
coefficients are the **mathematical backbone** of spin adaptation in CI, CSFs, and
functions like `configs_and_T_matrix`.


### 2.1. Physical meaning of Clebsch–Gordan coefficients
[Back to TOC](#TOC)
<a name="2.1"></a>

In quantum mechanics, CG coefficients describe how two angular momenta combine:

$$
\mathbf{J} = \mathbf{j}_1 + \mathbf{j}_2
$$

They provide the change of basis between:

- **Uncoupled basis**:
$$
| j_1 m_1 \rangle \otimes | j_2 m_2 \rangle
$$

- **Coupled basis**:
$$
| J M \rangle
$$

The overlap between these bases is:

$$
\boxed{
\langle j_1 m_1, j_2 m_2 \mid J M \rangle
}
$$

This number is the **Clebsch–Gordan coefficient**.


### 2.2. Why CG coefficients appear in electronic structure
[Back to TOC](#TOC)
<a name="2.2"></a>

In electronic structure theory:

- Each electron has spin
$$
s = \tfrac{1}{2}
$$
- Many-electron spin functions are constructed by **coupling spins pairwise**
- Determinants are eigenfunctions of $\hat{S}_z$, but **not** of $\hat{S}^2$
- CSFs *are* eigenfunctions of both $\hat{S}^2$ and $\hat{S}_z$

CG coefficients are the algebraic weights that convert determinants into CSFs.

### 2.3. Basis transformation
[Back to TOC](#TOC)
<a name="2.3"></a>

The key identity is:

$$
| J M \rangle
=
\sum_{m_1, m_2}
\langle j_1 m_1, j_2 m_2 \mid J M \rangle
\;
| j_1 m_1 \rangle | j_2 m_2 \rangle
$$

In CI language:

$$
| \text{CSF}_{S,M_s} \rangle
=
\sum_j
T_{j}^{(S,M_s)}
| D_j \rangle
$$

where the elements of $T$ are **products of Clebsch–Gordan coefficients**.

### 2.4. Selection rules (why many coefficients are zero)
[Back to TOC](#TOC)
<a name="2.4"></a>

The function explicitly enforces the selection rules:

#### Magnetic quantum numbers
$$
m_1 + m_2 = M
$$

#### Angular momentum bounds
$$
|j_1 - j_2| \le J \le j_1 + j_2
$$

#### Projection limits
$$
|m_1| \le j_1, \quad |m_2| \le j_2, \quad |M| \le J
$$

If any rule is violated, the coefficient is exactly:

$$
\langle j_1 m_1, j_2 m_2 \mid J M \rangle = 0
$$

This sparsity is crucial for efficient CSF construction.

### 2.5. Racah formula (explicit expression)
[Back to TOC](#TOC)
<a name="2.5"></a>

The function evaluates the CG coefficient using the **Racah summation formula**:

$$
\begin{aligned}
\langle j_1 m_1, j_2 m_2 | J M \rangle
&=
\delta_{M, m_1 + m_2}
(-1)^{j_1 - j_2 + M}
\sqrt{2J + 1}
\\
&\times
\sqrt{
\frac{(J + j_1 - j_2)! (J - j_1 + j_2)! (j_1 + j_2 - J)!}
     {(J + j_1 + j_2 + 1)!}
}
\\
&\times
\sqrt{
(J + M)! (J - M)!
(j_1 + m_1)! (j_1 - m_1)!
(j_2 + m_2)! (j_2 - m_2)!
}
\\
&\times
\sum_k
\frac{(-1)^k}{
k!
(j_1 + j_2 - J - k)!
(j_1 - m_1 - k)!
(j_2 + m_2 - k)!
}
\\
&\times
\frac{1}{
(J - j_2 + m_1 + k)!
(J - j_1 - m_2 + k)!
}
\end{aligned}
$$

The summation limits are chosen such that **all factorial arguments are non-negative**.

### 2.6. Half-integer spins and implementation detail
[Back to TOC](#TOC)
<a name="2.6"></a>

Electronic spins are half-integers:

$$
j = \tfrac{1}{2}, \tfrac{3}{2}, \dots
$$

To avoid fractional factorials, the implementation:

- Converts all angular momenta to **twice-values**:
$$
\tilde{j} = 2j \in \mathbb{Z}
$$
- Uses integer arithmetic internally
- Converts back to floating-point at the end

This ensures numerical stability without symbolic math.

### 2.7. Explicit spin-½ example (two electrons)
[Back to TOC](#TOC)
<a name="2.7"></a>

Each electron has:

$$
j_1 = j_2 = \tfrac{1}{2}
$$

#### Triplet ($S = 1$)

$$
\begin{aligned}
|1,1\rangle &= |\alpha \alpha\rangle \\
|1,0\rangle &= \tfrac{1}{\sqrt{2}}
(|\alpha \beta\rangle + |\beta \alpha\rangle) \\
|1,-1\rangle &= |\beta \beta\rangle
\end{aligned}
$$

Corresponding CG coefficients:

$$
\langle \tfrac{1}{2} \tfrac{1}{2}, \tfrac{1}{2} \tfrac{1}{2} | 1 1 \rangle = 1
$$

$$
\langle \tfrac{1}{2} \tfrac{1}{2}, \tfrac{1}{2} -\tfrac{1}{2} | 1 0 \rangle
=
\langle \tfrac{1}{2} -\tfrac{1}{2}, \tfrac{1}{2} \tfrac{1}{2} | 1 0 \rangle
=
\tfrac{1}{\sqrt{2}}
$$

#### Singlet ($S = 0$)

$$
|0,0\rangle
=
\tfrac{1}{\sqrt{2}}
(|\alpha \beta\rangle - |\beta \alpha\rangle)
$$

CG coefficients:

$$
\langle \tfrac{1}{2} \tfrac{1}{2}, \tfrac{1}{2} -\tfrac{1}{2} | 0 0 \rangle
=
\tfrac{1}{\sqrt{2}}
$$

$$
\langle \tfrac{1}{2} -\tfrac{1}{2}, \tfrac{1}{2} \tfrac{1}{2} | 0 0 \rangle
=
-\tfrac{1}{\sqrt{2}}
$$

These are exactly the coefficients used in singlet CSFs.

### 2.8. Relation to CSFs and the T matrix
[Back to TOC](#TOC)
<a name="2.8"></a>

In `configs_and_T_matrix`:

- Each CSF is built by recursively coupling spins
- At each coupling step, CG coefficients appear
- The final $T$ matrix is a **product of CG coefficients**

Schematically:

$$
T_{jI}
=
\prod_{\text{coupling steps}}
\langle j_a m_a, j_b m_b \mid J_{ab} M_{ab} \rangle
$$

Thus, `clebsch_gordan` is the **atomic primitive** of the entire spin-adaptation machinery.

### 2.9. Numerical considerations
[Back to TOC](#TOC)
<a name="2.9"></a>

- The function returns **real numbers** (spin CGs are real)
- Small values below `tol` are treated as zero
- Violating selection rules returns exactly `0.0`
- Phase conventions follow the **Condon–Shortley convention**

Consistency of this convention is crucial:  
changing it would flip signs in CSFs and CI matrices.


### 2.10. Main function
[Back to TOC](#TOC)
<a name="2.10"></a>
<a name="clebsch_gordan-1"></a>

In [12]:
help(cg.clebsch_gordan)

Help on function clebsch_gordan in module libra_py.citools.clebsch_gordan:

clebsch_gordan(j1: Union[int, float], j2: Union[int, float], J: Union[int, float], m1: Union[int, float], m2: Union[int, float], M: Union[int, float], tol: float = 1e-12) -> float
    Compute the Clebsch–Gordan coefficient ⟨j₁ m₁, j₂ m₂ | J M⟩ using the Racah formula.
    
    This function implements a purely numerical (no `sympy`) version of the Racah formula
    for Clebsch–Gordan coefficients, which describe the coupling of two angular momenta
    `j₁` and `j₂` to form a total angular momentum `J`. The result is a real-valued
    coefficient corresponding to the overlap between the uncoupled basis
    |j₁ m₁⟩ |j₂ m₂⟩ and the coupled basis |J M⟩.
    
    Parameters
    ----------
    j1, j2, J : int or float
        Angular momentum quantum numbers (can be integers or half-integers).
        Must satisfy the triangular condition:
        |j₁ − j₂| ≤ J ≤ j₁ + j₂.
    m1, m2, M : int or float
        Magnetic

In [13]:
cg.clebsch_gordan(1, 1, 2, 1, 1, 2)

0.9999999999999999

In [14]:
cg.clebsch_gordan(1, 1, 1, 1, 0, 1)

-0.7071067811865476

In [15]:
cg.clebsch_gordan(0.5, 0.5, 1, 0.5, -0.5, 0)

0.7071067811865476

In [16]:
cg.clebsch_gordan(0.5, 0.5, 0, 0.5, -0.5, 0)

0.7071067811865476

In [17]:
# 2 electrons  j1, j2, J, m1, m2, M : for |j1,m1>|j2,m2> -> |J,M>
print("Singlet coupling (Ms = 0): ")
print( cg.clebsch_gordan(0.5, 0.5, 0.0, 0.5,  0.5, 0.0) )  # |aa>
print( cg.clebsch_gordan(0.5, 0.5, 0.0, 0.5, -0.5, 0.0) )  # |ab>
print( cg.clebsch_gordan(0.5, 0.5, 0.0,-0.5,  0.5, 0.0) )  # |ba>
print( cg.clebsch_gordan(0.5, 0.5, 0.0,-0.5, -0.5, 0.0) )  # |bb>

print("Triplet coupling (Ms = 0): ")
print( cg.clebsch_gordan(0.5, 0.5, 1.0, 0.5,  0.5, 0.0) )  # |aa>
print( cg.clebsch_gordan(0.5, 0.5, 1.0, 0.5, -0.5, 0.0) )  # |ab>
print( cg.clebsch_gordan(0.5, 0.5, 1.0,-0.5,  0.5, 0.0) )  # |ba>
print( cg.clebsch_gordan(0.5, 0.5, 1.0,-0.5, -0.5, 0.0) )  # |bb>

print("Triplet coupling (Ms = -1): ")
print( cg.clebsch_gordan(0.5, 0.5, 1.0, 0.5,  0.5, -1.0) )  # |aa>
print( cg.clebsch_gordan(0.5, 0.5, 1.0, 0.5, -0.5, -1.0) )  # |ab>
print( cg.clebsch_gordan(0.5, 0.5, 1.0,-0.5,  0.5, -1.0) )  # |ba>
print( cg.clebsch_gordan(0.5, 0.5, 1.0,-0.5, -0.5, -1.0) )  # |bb>

print("Triplet coupling (Ms = 1): ")
print( cg.clebsch_gordan(0.5, 0.5, 1.0, 0.5,  0.5, 1.0) )  # |aa>
print( cg.clebsch_gordan(0.5, 0.5, 1.0, 0.5, -0.5, 1.0) )  # |ab>
print( cg.clebsch_gordan(0.5, 0.5, 1.0,-0.5,  0.5, 1.0) )  # |ba>
print( cg.clebsch_gordan(0.5, 0.5, 1.0,-0.5, -0.5, 1.0) )  # |bb>

Singlet coupling (Ms = 0): 
0.0
0.7071067811865476
-0.7071067811865476
0.0
Triplet coupling (Ms = 0): 
0.0
0.7071067811865476
0.7071067811865476
0.0
Triplet coupling (Ms = -1): 
0.0
0.0
0.0
-1.0000000000000002
Triplet coupling (Ms = 1): 
-1.0000000000000002
0.0
0.0
0.0


### 2.11. Helper functions
[Back to TOC](#TOC)
<a name="2.11"></a>
<a name="recursive_couple_spins_int-1"></a>

In [18]:
help(cg.recursive_couple_spins_int)

Help on function recursive_couple_spins_int in module libra_py.citools.clebsch_gordan:

recursive_couple_spins_int(spin_list_2: List[int]) -> Dict[Tuple[int, int], List[Tuple[Tuple[int, ...], float]]]
    Recursively couple a list of spin-½ particles using Clebsch–Gordan coefficients (integer 2*M convention).
    
    This function builds the total spin states for a system of N spin-½ particles 
    (each represented by +1 for α-spin and −1 for β-spin, corresponding to 2*M_i values).  
    The coupling is performed recursively: each additional spin-½ is combined with the 
    previously coupled subsystem using the Clebsch–Gordan coefficients to produce 
    total spin multiplets characterized by total spin S and total projection M.
    
    Results are expressed in the integer convention for twice-values:
    - `S2 = 2 * S`
    - `M2 = 2 * M`
    
    Each entry of the returned dictionary corresponds to a particular total spin 
    manifold (S2, M2) and contains all spin configurations

In [19]:
cg.recursive_couple_spins_int([+1])

{(1, 1): [((1,), 1.0)]}

In [20]:
cg.recursive_couple_spins_int([+1, -1])

{(0, 0): [((1, -1), -0.7071067811865476)],
 (2, 0): [((1, -1), 0.7071067811865476)]}

In [21]:
cg.recursive_couple_spins_int([+1, +1, -1])

{(1, 1): [((1, 1, -1), -0.7071067811865476),
  ((1, 1, -1), -0.7071067811865476),
  ((1, 1, -1), 0.408248290463863)],
 (3, 1): [((1, 1, -1), -0.5773502691896258)]}

## 3. Constructing configuration state functions (CSFs)
[Back to TOC](#TOC)
<a name="3"></a>

### 3.1. Main function
[Back to TOC](#TOC)
<a name="3.1"></a>
<a name="generate_CSFs_grouped-1"></a>

In [22]:
help(csf.generate_CSFs_grouped)

Help on function generate_CSFs_grouped in module libra_py.citools.csf:

generate_CSFs_grouped(dets_with_parity: Iterable[Tuple[Tuple[int, ...], int]], tol: float = 1e-12) -> Dict[Tuple[float, float], List[List[Tuple[Tuple[int, ...], float]]]]
    Build Configuration State Functions (CSFs) grouped by spatial-occupation patterns.
    
    This function constructs spin-adapted CSFs from a list of determinants 
    with their permutation parities. Determinants are first grouped by their 
    spatial orbital occupation patterns (e.g., which orbitals are doubly or singly 
    occupied), and then spin couplings are applied to unpaired electrons within 
    each group using Clebsch–Gordan coefficients.
    
    Parameters
    ----------
    dets_with_parity : iterable of (tuple[int, ...], int)
        Iterable of determinants and their canonicalization parities.
        - det: tuple of integers representing spin-orbitals, e.g., `(1, -2, 3, -4)`
        - parity: +1 or -1, the permutation parit

In [23]:
active_orbitals = [1,2]
N = 2
dets_with_parity = list(sd.generate_determinants_with_parity(active_orbitals, N))
csfs = csf.generate_CSFs_grouped(dets_with_parity)

print("Determinants:", dets_with_parity)
print("CSFS:", csfs)
print("\n=== Final CSFs ===")
csf.print_csfs(csfs)

Determinants: [((1, -1), 1), ((1, 2), 1), ((1, -2), 1), ((-1, 2), 1), ((-1, -2), 1), ((2, -2), 1)]
CSFS: {(0.0, 0.0): [[((1, -1), 1.0)], [((1, -2), 0.7071067811865476), ((-1, 2), -0.7071067811865476)], [((2, -2), 1.0)]], (1.0, -1.0): [[((-1, -2), 1.0)]], (1.0, 0.0): [[((1, -2), 0.7071067811865476), ((-1, 2), 0.7071067811865476)]], (1.0, 1.0): [[((1, 2), 1.0)]]}

=== Final CSFs ===
S,Ms = (0.0,0.0)
  CSF #1:
    (1, -1)  +1.000000
  CSF #2:
    (1, -2)  +0.707107
    (-1, 2)  -0.707107
  CSF #3:
    (2, -2)  +1.000000
S,Ms = (1.0,-1.0)
  CSF #1:
    (-1, -2)  +1.000000
S,Ms = (1.0,0.0)
  CSF #1:
    (1, -2)  +0.707107
    (-1, 2)  +0.707107
S,Ms = (1.0,1.0)
  CSF #1:
    (1, 2)  +1.000000


In [24]:
active_orbitals = [1,2,3,4]
N = 3
dets_with_parity = list(sd.generate_determinants_with_parity(active_orbitals, N))
csfs = csf.generate_CSFs_grouped(dets_with_parity)

print("Determinants:", dets_with_parity)
print("CSFS:", csfs)
print("\n=== Final CSFs ===")
csf.print_csfs(csfs)

Determinants: [((1, -1, 2), 1), ((1, -1, -2), 1), ((1, -1, 3), 1), ((1, -1, -3), 1), ((1, -1, 4), 1), ((1, -1, -4), 1), ((1, 2, -2), 1), ((1, 2, 3), 1), ((1, 2, -3), 1), ((1, 2, 4), 1), ((1, 2, -4), 1), ((1, -2, 3), 1), ((1, -2, -3), 1), ((1, -2, 4), 1), ((1, -2, -4), 1), ((1, 3, -3), 1), ((1, 3, 4), 1), ((1, 3, -4), 1), ((1, -3, 4), 1), ((1, -3, -4), 1), ((1, 4, -4), 1), ((-1, 2, -2), 1), ((-1, 2, 3), 1), ((-1, 2, -3), 1), ((-1, 2, 4), 1), ((-1, 2, -4), 1), ((-1, -2, 3), 1), ((-1, -2, -3), 1), ((-1, -2, 4), 1), ((-1, -2, -4), 1), ((-1, 3, -3), 1), ((-1, 3, 4), 1), ((-1, 3, -4), 1), ((-1, -3, 4), 1), ((-1, -3, -4), 1), ((-1, 4, -4), 1), ((2, -2, 3), 1), ((2, -2, -3), 1), ((2, -2, 4), 1), ((2, -2, -4), 1), ((2, 3, -3), 1), ((2, 3, 4), 1), ((2, 3, -4), 1), ((2, -3, 4), 1), ((2, -3, -4), 1), ((2, 4, -4), 1), ((-2, 3, -3), 1), ((-2, 3, 4), 1), ((-2, 3, -4), 1), ((-2, -3, 4), 1), ((-2, -3, -4), 1), ((-2, 4, -4), 1), ((3, -3, 4), 1), ((3, -3, -4), 1), ((3, 4, -4), 1), ((-3, 4, -4), 1)]
CSFS:

### 3.2. Helper functions
[Back to TOC](#TOC)
<a name="3.2"></a>
<a name="print_csfs-1"></a>

In [25]:
help(csf.print_csfs)

Help on function print_csfs in module libra_py.citools.csf:

print_csfs(csfs)
    Nicely print Configuration State Functions (CSFs) returned by `generate_CSFs_grouped`.
    
    This function iterates over all `(S, M)` sectors in the `csfs` dictionary
    and prints each CSF with its determinant(s) and corresponding coefficients.
    It handles both standard CSF lists `[(det, coeff), ...]` and accidental 
    tuple entries `(det, coeff)` safely.
    
    Parameters
    ----------
    csfs : dict[tuple[float, float], list[list[tuple[tuple[int, ...], float]]]]
        Dictionary of CSFs as returned by `generate_CSFs_grouped`.
        Keys are `(S, M)` total spin and projection.
        Values are lists of CSFs, each CSF is a list of `(det, coeff)` pairs.
    
    Prints
    ------
    For each `(S, M)` sector:
      - Line indicating the spin: "S,Ms = (S,M)"
      - Each CSF with index: "  CSF #i:"
      - Each determinant and its coefficient: "    det  coeff" with coeff formatted as `+/

<a name="group_key_from_det-1"></a>

In [26]:
help(csf.group_key_from_det)

Help on function group_key_from_det in module libra_py.citools.csf:

group_key_from_det(det)
    Compute a grouping key for a determinant based on orbital occupation.
    
    The function counts the number of electrons occupying each spatial orbital 
    in the determinant and separates orbitals into:
      - doubly occupied orbitals
      - singly occupied orbitals
    
    Parameters
    ----------
    det : tuple[int]
        Determinant represented as a tuple of spin-orbitals.
        Positive values indicate α-spin, negative values β-spin, and 
        the absolute value corresponds to the spatial orbital index.
        Example: `(1, -2, 2, -1)`.
    
    Returns
    -------
    tuple[tuple[int, ...], tuple[int, ...]]
        A tuple `(double_orbs, single_orbs)` where:
        - `double_orbs` : tuple of orbital indices that are doubly occupied (2 electrons)
        - `single_orbs` : tuple of orbital indices that are singly occupied (1 electron)
        
        The orbital indice

In [27]:
csf.group_key_from_det((1, -1, 2, -3))

((1,), (2, 3))

In [28]:
csf.group_key_from_det((1, 2, -2, 3, -3))

((2, 3), (1,))

<a name="build_unpaired_spinlist-1"></a>

In [29]:
help(csf.build_unpaired_spinlist)

Help on function build_unpaired_spinlist in module libra_py.citools.csf:

build_unpaired_spinlist(det, single_orbs)
    Construct a spin list for the unpaired electrons in a determinant.
    
    Given a determinant and a list of singly-occupied orbitals (`single_orbs`),
    this function returns a tuple of spin projections for the unpaired electrons
    in the specified orbital order. The spins are represented as integers:
        +1 → α-spin (spin up)
        -1 → β-spin (spin down)
    
    Parameters
    ----------
    det : tuple[int]
        Determinant represented as a tuple of spin-orbitals. 
        Positive values indicate α-spin, negative values β-spin.
        Example: `(1, -2, 3, -4)`.
    single_orbs : iterable[int]
        Ordered list of spatial orbital indices that are singly occupied in this determinant.
        The order determines the order of spins in the returned tuple.
    
    Returns
    -------
    tuple[int, ...]
        Tuple of +1/-1 values corresponding to

In [30]:
csf.build_unpaired_spinlist((1, -2, 3), [2, 3])

(-1, 1)

In [31]:
csf.build_unpaired_spinlist((1, 2, -3), [2, 3])

(1, -1)